In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv('/content/diabetes.csv')

In [ ]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [ ]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [ ]:
X=df.iloc[ :,:-1].values
y=df.iloc[ :,-1].values

In [ ]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()


In [ ]:
X=sc.fit_transform(X)

In [ ]:
X.shape

(768, 8)

In [ ]:
from sklearn.model_selection import train_test_split



In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [ ]:
# Build
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(1, activation='sigmoid'))

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6016 - loss: 0.6565 - val_accuracy: 0.6623 - val_loss: 0.6159
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6205 - loss: 0.6299 - val_accuracy: 0.7078 - val_loss: 0.5832
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6948 - loss: 0.5749 - val_accuracy: 0.7273 - val_loss: 0.5593
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7408 - loss: 0.5320 - val_accuracy: 0.7468 - val_loss: 0.5426
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7226 - loss: 0.5486 - val_accuracy: 0.7727 - val_loss: 0.5304
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7714 - loss: 0.4898 - val_accuracy: 0.7792 - val_loss: 0.5223
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7193 - loss: 0.5202 - val_accuracy: 0.7922 - val_loss: 0.5151
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7265 - loss: 0.5230 - val_accuracy: 0.7987 - val_loss

In [ ]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 1.6 MB/s eta 0:00:00


In [ ]:
import keras_tuner as kt

In [ ]:
def build_model(hp):

    model = Sequential()

    model.add(Dense(32, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta'])

    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model

In [ ]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5)

tuner.search(X_train, y_train, epochs=5,
             validation_data=(X_test, y_test))

Trial 4 Complete [00h 00m 02s]
val_accuracy: 0.7727272510528564

Best val_accuracy So Far: 0.8246753215789795
Total elapsed time: 00h 00m 10s


In [ ]:
tuner.results_summary()

Results summary
Results in ./untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0 summary
Hyperparameters:
optimizer: adam
Score: 0.8246753215789795

Trial 3 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.7727272510528564

Trial 1 summary
Hyperparameters:
optimizer: sgd
Score: 0.7402597665786743

Trial 2 summary
Hyperparameters:
optimizer: adadelta
Score: 0.5844155550003052


In [ ]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(X_train, y_train,
          batch_size=32,
          epochs=100,
          initial_epoch=6,
          validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7609 - loss: 0.5315 - val_accuracy: 0.7922 - val_loss: 0.4995
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7676 - loss: 0.5119 - val_accuracy: 0.8052 - val_loss: 0.4859
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7531 - loss: 0.5217 - val_accuracy: 0.8052 - val_loss: 0.4750
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8011 - loss: 0.4857 - val_accuracy: 0.8052 - val_loss: 0.4658
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8104 - loss: 0.4748 - val_accuracy: 0.8052 - val_loss: 0.4595
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7742 - loss: 0.4842 - val_accuracy: 0.7987 - val_loss: 0.4556
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7763 - loss: 0.4761 - val_accuracy: 0.7987 - val_loss: 0.4516
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7661 - loss: 0.4745 - val_accuracy: 0.79

In [ ]:
def build_model(hp):

    model = Sequential()

    units = hp.Int('units', min_value=8, max_value=128, step=8)

    model.add(Dense(units=units, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='rmsprop',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model

In [ ]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5,
                        directory ='mydir')

tuner.search(X_train, y_train, epochs=5,
             validation_data=(X_test, y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.7467532753944397

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 13s


In [ ]:
tuner.results_summary()

Results summary
Results in mydir/untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 2 summary
Hyperparameters:
units: 88
Score: 0.8051947951316833

Trial 0 summary
Hyperparameters:
units: 64
Score: 0.7857142686843872

Trial 3 summary
Hyperparameters:
units: 120
Score: 0.7792207598686218

Trial 1 summary
Hyperparameters:
units: 56
Score: 0.7727272510528564

Trial 4 summary
Hyperparameters:
units: 24
Score: 0.7467532753944397


In [ ]:
tuner.get_best_hyperparameters()[0].values

{'units': 88}

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 88)             │           792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            89 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 881 (3.44 KB)

 Trainable params: 881 (3.44 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(X_train, y_train,
          batch_size=32,
          epochs=100,
          initial_epoch=6,
          validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.7697 - loss: 0.5054 - val_accuracy: 0.7792 - val_loss: 0.4740
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7624 - loss: 0.4927 - val_accuracy: 0.7857 - val_loss: 0.4651
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7648 - loss: 0.4936 - val_accuracy: 0.7987 - val_loss: 0.4602
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7882 - loss: 0.4442 - val_accuracy: 0.7922 - val_loss: 0.4569
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7956 - loss: 0.4456 - val_accuracy: 0.7922 - val_loss: 0.4552
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7969 - loss: 0.4378 - val_accuracy: 0.7922 - val_loss: 0.4547
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7574 - loss: 0.4762 - val_accuracy: 0.7922 - val_loss: 0.4544
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7599 - loss: 0.4659 - val_accuracy: 0.7

In [ ]:
def build_model(hp):

    model = Sequential()

    model.add(Dense(72, activation='relu', input_dim=8))

    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
        model.add(Dense(72, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='rmsprop',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model



In [ ]:

tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=3,
                        directory='mydir',
                        project_name='layers')

In [ ]:
tuner.search(X_train, y_train, epochs=5,
             validation_data=(X_test, y_test))

Trial 3 Complete [00h 00m 04s]
val_accuracy: 0.8246753215789795

Best val_accuracy So Far: 0.8246753215789795
Total elapsed time: 00h 00m 11s


In [ ]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 2}

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
model.fit(X_train, y_train,
          batch_size=32,
          epochs=100,
          initial_epoch=6,
          validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7657 - loss: 0.4863 - val_accuracy: 0.8052 - val_loss: 0.4801
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7579 - loss: 0.4769 - val_accuracy: 0.8117 - val_loss: 0.4757
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7890 - loss: 0.4377 - val_accuracy: 0.8117 - val_loss: 0.4742
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8112 - loss: 0.4297 - val_accuracy: 0.8052 - val_loss: 0.4817
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7929 - loss: 0.4446 - val_accuracy: 0.7987 - val_loss: 0.4713
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7938 - loss: 0.4393 - val_accuracy: 0.8117 - val_loss: 0.4935
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7580 - loss: 0.4479 - val_accuracy: 0.7922 - val_loss: 0.4750
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8196 - loss: 0.4077 - val_accuracy: 0.78

In [ ]:
def build_model(hp):

    model = Sequential()

    counter = 0

    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):

        if counter == 0:
            model.add(
                Dense(
                    hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                    activation=hp.Choice('activation' + str(i),
                                         values=['relu', 'tanh', 'sigmoid']),
                    input_dim=8
                )
            )
            model.add(
    Dropout(
        hp.Choice('dropout' + str(i),
                  values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
    )
)
        else:
            model.add(
                Dense(
                    hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                    activation=hp.Choice('activation' + str(i),
                                         values=['relu', 'tanh', 'sigmoid'])
                )
            )
            model.add(
    Dropout(
        hp.Choice('dropout' + str(i),
                  values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
    )
)
        counter += 1

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice('optimizer',
                            values=['rmsprop', 'adam', 'sgd', 'nadam', 'adadelta']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=3,
    directory="mydir",
    project_name='final'
)

In [ ]:
tuner.search(X_train, y_train, epochs=5,
             validation_data=(X_test, y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.7857142686843872
Total elapsed time: 00h 00m 13s


In [ ]:
tuner.results_summary()

Results summary
Results in mydir/final
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 1 summary
Hyperparameters:
num_layers: 2
units0: 64
activation0: tanh
dropout0: 0.7
optimizer: rmsprop
units1: 56
activation1: tanh
dropout1: 0.5
units2: 64
activation2: relu
dropout2: 0.6
units3: 120
activation3: tanh
dropout3: 0.4
units4: 48
activation4: relu
dropout4: 0.2
units5: 96
activation5: tanh
dropout5: 0.8
units6: 88
activation6: tanh
dropout6: 0.2
units7: 96
activation7: relu
dropout7: 0.9
Score: 0.7857142686843872

Trial 0 summary
Hyperparameters:
num_layers: 8
units0: 48
activation0: relu
dropout0: 0.2
optimizer: adam
units1: 8
activation1: relu
dropout1: 0.1
units2: 8
activation2: relu
dropout2: 0.1
units3: 8
activation3: relu
dropout3: 0.1
units4: 8
activation4: relu
dropout4: 0.1
units5: 8
activation5: relu
dropout5: 0.1
units6: 8
activation6: relu
dropout6: 0.1
units7: 8
activation7: relu
dropout7: 0.1
Score: 0.7467532753944397

Trial 2 summary
Hyperpar

In [ ]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 2,
 'units0': 64,
 'activation0': 'tanh',
 'dropout0': 0.7,
 'optimizer': 'rmsprop',
 'units1': 56,
 'activation1': 'tanh',
 'dropout1': 0.5,
 'units2': 64,
 'activation2': 'relu',
 'dropout2': 0.6,
 'units3': 120,
 'activation3': 'tanh',
 'dropout3': 0.4,
 'units4': 48,
 'activation4': 'relu',
 'dropout4': 0.2,
 'units5': 96,
 'activation5': 'tanh',
 'dropout5': 0.8,
 'units6': 88,
 'activation6': 'tanh',
 'dropout6': 0.2,
 'units7': 96,
 'activation7': 'relu',
 'dropout7': 0.9}

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 8 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 56)             │         3,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            57 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,273 (16.69 KB)

 Trainable params: 4,273 (16.69 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(X_train, y_train,
          batch_size=32,
          epochs=100,
          initial_epoch=6,
          validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7341 - loss: 0.5379 - val_accuracy: 0.7922 - val_loss: 0.4707
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7221 - loss: 0.5404 - val_accuracy: 0.7987 - val_loss: 0.4698
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7666 - loss: 0.4907 - val_accuracy: 0.7857 - val_loss: 0.4669
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7345 - loss: 0.5546 - val_accuracy: 0.7987 - val_loss: 0.4650
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7253 - loss: 0.5314 - val_accuracy: 0.7857 - val_loss: 0.4646
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7457 - loss: 0.5519 - val_accuracy: 0.7922 - val_loss: 0.4640
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7427 - loss: 0.5278 - val_accuracy: 0.7857 - val_loss: 0.4679
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7181 - loss: 0.5323 - val_accuracy